# Single question RAG retrieval probe

Put one question in `QUESTION`, run the cells, and inspect the full RAG path: query processing, retrieved chunks, aggregated sections, selector/context, displayed sources, and final answer.

This notebook uses the same runtime RAG config mapping as `apps/streamlit-ui/pages/01_Chatbot.py` for the selected staging/prod database.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from typing import Any

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from psycopg.rows import dict_row


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "packages/rag-pipeline").exists():
            return candidate
    raise RuntimeError("Could not find assistant-rh repository root from current working directory.")


REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env")

for package_src in (REPO_ROOT / "packages/rag-pipeline/src", REPO_ROOT / "packages/shared-config/src"):
    package_src_str = str(package_src)
    if package_src_str not in sys.path:
        sys.path.insert(0, package_src_str)

print(f"Repo root: {REPO_ROOT}")


## Controls

Edit only `QUESTION` for the usual workflow. Set `EXPECTED_SOURCE_FAMILY` only if you want green highlighting for a target source family.

In [ ]:
TARGET_ENV = "staging"  # "staging" or "prod"
QUESTION = "Quel est le montant de la prise en charge des frais de transports en commun domicile-travail pour un agent public ?"

# Optional highlighting: "", "service_public", "dgafp_legifrance", "matte", or "rgrh".
EXPECTED_SOURCE_FAMILY = "service_public"

TOP_N_CHUNKS = 20
TOP_N_SECTIONS = 12
DB_CONNECT_TIMEOUT_S = 10
AUTO_DISABLE_MISSING_OPTIONAL_TABLES = True
SHOW_RAW_STAGE_TRACE = False

DSN_ENV_BY_TARGET = {
    "staging": "SCW_POSTGRES_DSN_STAGING",
    "prod": "SCW_POSTGRES_DSN_PROD",
}


def with_connect_timeout(dsn: str, seconds: int) -> str:
    if "connect_timeout=" in dsn:
        return dsn
    if "://" not in dsn:
        return f"{dsn} connect_timeout={seconds}"
    separator = "&" if "?" in dsn else "?"
    return f"{dsn}{separator}connect_timeout={seconds}"


def resolve_dsn(target_env: str) -> tuple[str, str]:
    key = DSN_ENV_BY_TARGET.get(target_env)
    if not key:
        raise ValueError(f"Unsupported TARGET_ENV={target_env!r}; expected one of {sorted(DSN_ENV_BY_TARGET)}")
    value = os.getenv(key, "").strip()
    source = key
    if not value and target_env == "staging":
        value = os.getenv("SCW_POSTGRES_DSN", "").strip()
        source = "SCW_POSTGRES_DSN fallback for staging"
    if not value:
        raise RuntimeError(f"Missing {key}. Add it to .env or the notebook kernel environment.")
    return with_connect_timeout(value, DB_CONNECT_TIMEOUT_S), source


def validate_provider_env() -> None:
    required = ["ALBERT_API_KEY", "ALBERT_BASE_URL", "SCALEWAY_API_KEY", "SCALEWAY_BASE_URL"]
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Missing provider env vars for full pipeline run: {', '.join(missing)}")


SELECTED_DSN, SELECTED_DSN_SOURCE = resolve_dsn(TARGET_ENV)
os.environ["APP_DB_TARGET"] = "scaleway"
os.environ["APP_SCALEWAY_ENV"] = TARGET_ENV
os.environ["SCW_POSTGRES_DSN"] = SELECTED_DSN
validate_provider_env()

print(f"Target: {TARGET_ENV} ({SELECTED_DSN_SOURCE}, connect_timeout={DB_CONNECT_TIMEOUT_S}s)")
print(f"Question: {QUESTION}")


## Build pipeline config

This mirrors the Streamlit V3 config mapping, then applies one notebook-only safety override: if `rag_chunks_test` is enabled in runtime config but absent from the selected DB, disable it to avoid repeated optional-table errors during inspection.

In [ ]:
from assistant_rh_rag_pipeline import create_pipeline, get_default_config
from assistant_rh_rag_pipeline.admin import get_rag_config
from assistant_rh_rag_pipeline.config import ContextMode, SearchMode


def build_streamlit_config(runtime_config: Any):
    """Mirror apps/streamlit-ui/pages/01_Chatbot.py RAG V3 config mapping."""
    config = get_default_config()
    mode_map = {"standard": ContextMode.STANDARD, "wide": ContextMode.WIDE}
    config.context.context_mode = mode_map.get(getattr(runtime_config, "v3_context_mode", "standard"), ContextMode.STANDARD)
    config.context.token_budget = getattr(runtime_config, "v3_token_budget", 8000)
    config.context.doc_entire_threshold = getattr(runtime_config, "v3_doc_entire_threshold", 3500)
    config.selector.enabled = getattr(runtime_config, "v3_enable_selector", True)
    config.selector.model = getattr(runtime_config, "v3_selector_model", "openweight-large")
    config.selector.prompt_name = getattr(runtime_config, "v3_selector_prompt_name", "v3_selector_business.md")
    config.query_processor.enable_intent_gating = getattr(runtime_config, "enable_intent_gating", False)
    config.query_processor.enable_acronym_expansion = getattr(runtime_config, "enable_query_expansion", True)
    config.query_processor.intent_prompt_name = getattr(runtime_config, "v3_intent_prompt_name", "intent_unified.md")
    config.retrieval.tables = list(getattr(runtime_config, "v3_tables", None) or ["matte", "service_public", "dgafp", "rgrh"])
    config.retrieval.enable_chunks_test = getattr(runtime_config, "v3_enable_chunks_test", True)
    config.retrieval.initial_top_k = getattr(runtime_config, "v3_initial_top_k", 10)
    config.retrieval.alpha = getattr(runtime_config, "v3_alpha", 0.5)
    config.aggregation.enable_section_reranker = getattr(runtime_config, "v3_enable_reranker", True)
    config.aggregation.section_rerank_top_k = getattr(runtime_config, "v3_rerank_top_k", 5)
    search_mode_map = {"semantic": SearchMode.SEMANTIC, "hybrid": SearchMode.HYBRID, "lexical": SearchMode.LEXICAL}
    config.retrieval.search_mode = search_mode_map.get(getattr(runtime_config, "v3_search_mode", "semantic"), SearchMode.SEMANTIC)
    config.generation.model = getattr(runtime_config, "v3_generator_model", "openweight-large")
    config.generation.temperature = getattr(runtime_config, "v3_temperature", 0.0)
    config.generation.system_prompt_name = getattr(runtime_config, "v3_system_prompt_name", "system_prompt_V6_optimized.md")
    config.verbose = getattr(runtime_config, "verbose_mode", False)
    return config


def table_exists(dsn: str, table_name: str) -> bool:
    with psycopg.connect(dsn, row_factory=dict_row) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT to_regclass(%s) AS relation", (f"public.{table_name}",))
            row = cur.fetchone()
    return bool(row and row["relation"])


def apply_notebook_safety_overrides(config: Any) -> list[str]:
    overrides: list[str] = []
    if AUTO_DISABLE_MISSING_OPTIONAL_TABLES and config.retrieval.enable_chunks_test and not table_exists(SELECTED_DSN, "rag_chunks_test"):
        config.retrieval.enable_chunks_test = False
        overrides.append(
            "Disabled retrieval.enable_chunks_test because runtime config enables rag_chunks_test, "
            "but public.rag_chunks_test is absent on the selected database."
        )
    return overrides


RUNTIME_RAG_CONFIG = get_rag_config()
PIPELINE_CONFIG = build_streamlit_config(RUNTIME_RAG_CONFIG)
NOTEBOOK_CONFIG_OVERRIDES = apply_notebook_safety_overrides(PIPELINE_CONFIG)
for override in NOTEBOOK_CONFIG_OVERRIDES:
    print(f"Notebook safety override: {override}")
PIPELINE = create_pipeline(PIPELINE_CONFIG, dsn=SELECTED_DSN)

display({
    "target_env": TARGET_ENV,
    "runtime_config": RUNTIME_RAG_CONFIG.to_dict(),
    "notebook_config_overrides": NOTEBOOK_CONFIG_OVERRIDES,
    "pipeline_config": PIPELINE_CONFIG.to_dict(),
})


## Run question

In [ ]:
started = time.perf_counter()
RESULT = PIPELINE.run_with_trace(QUESTION)
WALL_MS = (time.perf_counter() - started) * 1000
METADATA = RESULT.metadata or {}

display(Markdown("## Query processor"))
display(pd.DataFrame([{
    "original_query": METADATA.get("original_query"),
    "query_for_retrieval": METADATA.get("query_for_retrieval"),
    "intent": METADATA.get("intent"),
    "theme": METADATA.get("theme"),
    "needs_legal_search": METADATA.get("needs_legal_search"),
    "tables_searched": ", ".join(METADATA.get("tables_searched") or []),
    "selector_decision": METADATA.get("selector_decision"),
    "selector_retry_triggered": METADATA.get("selector_retry_triggered"),
    "wall_ms": round(WALL_MS, 1),
}]))

display(Markdown("## Timing"))
display(pd.DataFrame([RESULT.timing or {}]).T.rename(columns={0: "ms_or_value"}))


## Stage views

In [ ]:
EXPECTED_TERMS = {
    "dgafp_legifrance": ("dgafp", "legifrance", "légifrance"),
    "matte": ("matte",),
    "service_public": ("service-public", "service_public", "service public"),
    "rgrh": ("rgrh",),
}


def compact_text(value: Any, limit: int = 480) -> str:
    text = " ".join(str(value or "").split())
    return text if len(text) <= limit else f"{text[: limit - 1]}…"


def matches_expected_family(row: dict[str, Any]) -> bool:
    if not EXPECTED_SOURCE_FAMILY:
        return False
    terms = EXPECTED_TERMS.get(EXPECTED_SOURCE_FAMILY, ())
    haystack = " ".join(str(row.get(key, "") or "") for key in ("table", "source_table", "publisher", "document_title")).lower()
    return any(term in haystack for term in terms)


def show_table(rows: list[dict[str, Any]], columns: list[str], max_rows: int | None = None) -> None:
    if not rows:
        display(Markdown("_No rows._"))
        return
    df = pd.DataFrame(rows)
    if max_rows is not None:
        df = df.head(max_rows)
    df = df[[column for column in columns if column in df.columns]]
    if "expected_hit" in df.columns and EXPECTED_SOURCE_FAMILY:
        display(df.style.apply(lambda row: ["background-color: #e8f5e9" if row.get("expected_hit") else "" for _ in row], axis=1))
    else:
        display(df)


attempts = [attempt for attempt in (METADATA.get("retrieval_attempts") or []) if isinstance(attempt, dict)]

for attempt in attempts:
    display(Markdown(f"### Retrieval attempt: `{attempt.get('name')}`"))
    display(pd.DataFrame([{
        "search_mode": attempt.get("search_mode"),
        "top_k": attempt.get("top_k"),
        "tables_searched": ", ".join(attempt.get("tables_searched") or []),
        "retrieved_chunks": len(attempt.get("retrieved_chunks") or []),
        "sections_before_rerank": (attempt.get("aggregation") or {}).get("sections_before_rerank"),
        "sections_after_rerank": (attempt.get("aggregation") or {}).get("sections_after_rerank"),
        "selector_items_before": (attempt.get("selector") or {}).get("items_before"),
        "selector_items_after": (attempt.get("selector") or {}).get("items_after"),
        "selector_all_rejected": (attempt.get("selector") or {}).get("all_rejected"),
    }]))

    chunk_rows = []
    for rank, chunk in enumerate(attempt.get("retrieved_chunks") or [], start=1):
        row = {
            "rank": rank,
            "score": chunk.get("score"),
            "table": chunk.get("table") or chunk.get("source_table") or "",
            "chunk_id": chunk.get("chunk_id", ""),
            "section_id": chunk.get("section_id", ""),
            "retrieval_path": chunk.get("retrieval_path", ""),
            "heading_match_score": chunk.get("heading_match_score"),
            "preview": compact_text(chunk.get("preview"), 520),
        }
        row["expected_hit"] = matches_expected_family(row)
        chunk_rows.append(row)
    display(Markdown("#### Retrieved chunks"))
    show_table(chunk_rows, ["rank", "expected_hit", "score", "table", "retrieval_path", "heading_match_score", "chunk_id", "section_id", "preview"], TOP_N_CHUNKS)

    section_rows = []
    for rank, section in enumerate(attempt.get("aggregated_sections") or [], start=1):
        row = {
            "rank": rank,
            "score": section.get("score"),
            "publisher": section.get("publisher", ""),
            "heading": section.get("heading", ""),
            "chunk_count": section.get("chunk_count"),
            "token_estimate": section.get("token_estimate"),
            "section_id": section.get("section_id", ""),
            "document_id": section.get("document_id", ""),
        }
        row["expected_hit"] = matches_expected_family(row)
        section_rows.append(row)
    display(Markdown("#### Aggregated sections"))
    show_table(section_rows, ["rank", "expected_hit", "score", "publisher", "heading", "chunk_count", "token_estimate", "section_id", "document_id"], TOP_N_SECTIONS)

    selector = attempt.get("selector") or {}
    reason = selector.get("reason") or selector.get("rejection_reason") or ""
    if reason:
        display(Markdown("#### Selector reasoning"))
        display(Markdown(compact_text(reason, 1500)))


## Final context, sources, answer

In [ ]:
context_rows = []
for rank, item in enumerate(RESULT.context_items or [], start=1):
    metadata = getattr(item, "metadata", {}) or {}
    row = {
        "rank": rank,
        "score": round(float(getattr(item, "score", 0.0) or 0.0), 4),
        "publisher": getattr(item, "publisher", "") or "",
        "heading": getattr(item, "heading", "") or "",
        "document_title": getattr(item, "document_title", "") or "",
        "tokens": getattr(item, "token_estimate", 0),
        "section_id": str(getattr(item, "section_id", "") or ""),
        "doc_id": str(metadata.get("doc_id", "") or ""),
        "is_doc_entire": bool(metadata.get("is_doc_entire", False)),
        "preview": compact_text(getattr(item, "content", ""), 700),
    }
    row["expected_hit"] = matches_expected_family(row)
    context_rows.append(row)

display(Markdown("### Final context items"))
show_table(context_rows, ["rank", "expected_hit", "score", "publisher", "heading", "document_title", "tokens", "is_doc_entire", "section_id", "preview"], None)

source_rows = []
for rank, source in enumerate(RESULT.sources or [], start=1):
    row = {
        "rank": rank,
        "publisher": source.get("publisher", ""),
        "heading": source.get("heading", ""),
        "document_title": source.get("document_title", ""),
        "document_url": source.get("document_url", ""),
        "score": source.get("score"),
    }
    row["expected_hit"] = matches_expected_family(row)
    source_rows.append(row)

display(Markdown("### Displayed sources"))
show_table(source_rows, ["rank", "expected_hit", "score", "publisher", "heading", "document_title", "document_url"], None)

display(Markdown("### Answer"))
display(Markdown(RESULT.answer or "_Empty answer._"))

if SHOW_RAW_STAGE_TRACE:
    display(Markdown("### Raw stage trace"))
    display(JSON(METADATA.get("stage_trace") or {}))
